# 03 - Train Baselines V15

This clean notebook separates two different thesis artifacts: V15 baseline model comparison and ensemble candidate prediction generation. It is demo-friendly by default and does not retrain models unless optional heavy cells are explicitly enabled.


## Section 1 - Purpose and distinction

This notebook has two purposes:

1. Train and compare V15 baseline models.
2. Prepare or summarize OOF prediction candidates for the ensemble stage.

Baseline models are for fair algorithm-level comparison on the same final V15 feature set. Candidate predictors are OOF prediction vectors used in the ensemble library and may come from strong historical models or stacked predictors.

Final thesis-facing terminology:

- SPC = Stacked Prediction Candidate = `fp_final`.
- V15 Multi-Seed LightGBM = `lgb_v15_ms`.
- SE-HC = Stacked Ensemble with Hill-Climbing Selection = `SIGMA_FINAL`.
- Final model formula: `SE-HC = 0.5 * SPC + 0.5 * V15 Multi-Seed LightGBM`.
- `SE_HC_SELECTED_CANDIDATES` and `SE_HC_SELECTED_CANDIDATES_MS` are additional experiments only, not final thesis results.


## Section 2 - V15 data and common CV setup

This section defines the shared V15 matrix paths, target extraction conventions, categorical columns, 5-fold stratified CV setup, and output paths. Full V15 parquet matrices are local-only artifacts and are not committed to GitHub.


In [ ]:

from pathlib import Path
import json
import os
import time
import gc
import warnings

import numpy as np
import pandas as pd
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import SGDClassifier
from sklearn.neural_network import MLPClassifier

warnings.filterwarnings('ignore')

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = ROOT / 'processed_train_test'
OOF_DIR = ROOT / 'outputs' / 'oof_predictions'
METRICS_DIR = ROOT / 'evaluation' / 'metrics'
REPORTS_DIR = ROOT / 'reports' / 'results'
AUDIT_DIR = ROOT / 'audit'

TRAIN_FILE = DATA_DIR / 'train_merged_v15.parquet'
TEST_FILE = DATA_DIR / 'test_merged_v15.parquet'
TARGET = 'TARGET'
ID_COL = 'SK_ID_CURR'
EXCLUDE_COLS = [ID_COL, TARGET, 'index']
N_FOLDS = 5
SEED = 42

FORCE_CATEGORICAL = [
    'OCCUPATION_TYPE',
    'ORGANIZATION_TYPE',
    'NAME_EDUCATION_TYPE',
    'NAME_INCOME_TYPE',
    'NAME_FAMILY_STATUS',
    'NAME_HOUSING_TYPE',
    'NAME_CONTRACT_TYPE',
    'CODE_GENDER',
    'WEEKDAY_APPR_PROCESS_START',
    'HOUR_APPR_PROCESS_START',
    'BURO_CLU_LABEL',
    'PREV_CLU_LABEL',
]

BASELINE_OOF_FILES = {
    'Logistic Regression': ('oof_lr_v15_baseline.parquet', 'oof_lr'),
    'MLP': ('oof_mlp_v15.parquet', 'oof_mlp'),
    'LightGBM single seed': ('oof_lgb_v15_single_seed.parquet', 'oof_lgb'),
    'LightGBM multi-seed': ('oof_lgb_v15_multiseed.parquet', 'oof_lgb'),
    'XGBoost': ('oof_xgb_v15.parquet', 'oof_xgb'),
    'CatBoost': ('oof_cb_v15.parquet', 'oof_cb'),
}

CANDIDATE_OOF_FILES = {
    'SPC / fp_final': ('oof_FP_FINAL.parquet', 'oof_final'),
    'lgb_v15_ms': ('oof_lgb_v15_multiseed.parquet', 'oof_lgb'),
    'lgb_v12_ms': ('oof_lgb_v12_multiseed.parquet', 'oof_lgb'),
    'lgb_v11_ms': ('oof_lgb_v11_multiseed.parquet', 'oof_lgb'),
    'lgb_v7_ms': ('oof_lgb_v7_multiseed.parquet', 'oof_lgb'),
    'cb_v11_ms': ('oof_cb_v11_multiseed.parquet', 'oof_cb'),
    'xgb_v11': ('oof_xgb_v11.parquet', 'oof_xgb'),
    'stack_v8 / K3_stack_v8_FINAL': ('oof_K3_stack_v8_FINAL.parquet', 'oof_final'),
    'V15 XGBoost': ('oof_xgb_v15.parquet', 'oof_xgb'),
    'V15 CatBoost': ('oof_cb_v15.parquet', 'oof_cb'),
}

print(f'ROOT={ROOT}')
print(f'TRAIN_FILE exists={TRAIN_FILE.exists()}')
print(f'TEST_FILE exists={TEST_FILE.exists()}')
print(f'OOF_DIR exists={OOF_DIR.exists()}')


ROOT=C:\Users\Admin\OneDrive - Beta Group\KLTN\Home Credit Defalt Risk
TRAIN_FILE exists=True
TEST_FILE exists=True
OOF_DIR exists=True


In [ ]:

def inspect_v15_metadata() -> dict:
    info_path = AUDIT_DIR / 'v15_feature_info.json'
    if info_path.exists():
        info = json.loads(info_path.read_text(encoding='utf-8'))
        # Normalize old absolute paths to repo-relative paths for GitHub readability.
        info['train_file'] = str(TRAIN_FILE.relative_to(ROOT)) if TRAIN_FILE.exists() else 'processed_train_test/train_merged_v15.parquet'
        info['test_file'] = str(TEST_FILE.relative_to(ROOT)) if TEST_FILE.exists() else 'processed_train_test/test_merged_v15.parquet'
        info['cv'] = 'StratifiedKFold(n_splits=5, shuffle=True, random_state=42)'
        return info

    if TRAIN_FILE.exists() and TEST_FILE.exists():
        import pyarrow.parquet as pq
        train_pf = pq.ParquetFile(TRAIN_FILE)
        test_pf = pq.ParquetFile(TEST_FILE)
        columns = train_pf.schema.names
        return {
            'train_file': str(TRAIN_FILE.relative_to(ROOT)),
            'test_file': str(TEST_FILE.relative_to(ROOT)),
            'train_shape': [train_pf.metadata.num_rows, len(columns)],
            'test_shape': [test_pf.metadata.num_rows, len(test_pf.schema.names)],
            'n_model_features': len([c for c in columns if c not in EXCLUDE_COLS]),
            'excluded_columns': EXCLUDE_COLS,
            'cv': 'StratifiedKFold(n_splits=5, shuffle=True, random_state=42)',
        }

    return {
        'train_file': 'processed_train_test/train_merged_v15.parquet',
        'test_file': 'processed_train_test/test_merged_v15.parquet',
        'train_shape': [307507, 1043],
        'test_shape': [48744, 1042],
        'n_model_features': 1041,
        'excluded_columns': EXCLUDE_COLS,
        'cv': 'StratifiedKFold(n_splits=5, shuffle=True, random_state=42)',
        'note': 'Local V15 parquet artifacts are gitignored and may be absent in a fresh GitHub clone.',
    }

inspect_v15_metadata()


{'train_file': 'processed_train_test/train_merged_v15.parquet', 'test_file': 'processed_train_test/test_merged_v15.parquet', 'train_shape': [307507, 1043], 'test_shape': [48744, 1042], 'n_model_features': 1041, 'excluded_columns': ['SK_ID_CURR', 'TARGET', 'index'], 'target_counts': {'0': 282682, '1': 24825}, 'cv': 'StratifiedKFold(n_splits=5, shuffle=True, random_state=42)'}

## Section 3 - V15 baseline model training

These models are trained on the same final V15 feature representation and are used for algorithm-level comparison.

Optional heavy cells: do not run during demo unless local artifacts and sufficient runtime are available. By default, the notebook reuses saved OOF files under `outputs/oof_predictions/`.


### Shared training helpers

Source: `src/train_v15_baselines.py`, adapted for notebook-safe paths.


In [ ]:
def log(message: str) -> None:
    print(message, flush=True)


def elapsed(start: float) -> str:
    return f"{(time.time() - start) / 60:.1f} min"


def load_v15() -> tuple[pd.DataFrame, pd.DataFrame, list[str], np.ndarray]:
    log("Loading existing V15 matrices...")
    train = pd.read_parquet(TRAIN_FILE)
    test = pd.read_parquet(TEST_FILE)
    feature_cols = [c for c in train.columns if c not in EXCLUDE_COLS]

    missing_in_test = sorted(set(feature_cols) - set(test.columns))
    if missing_in_test:
        raise ValueError(f"Feature columns missing in test: {missing_in_test[:20]}")

    log(f"Train shape: {train.shape}")
    log(f"Test shape:  {test.shape}")
    log(f"Model feature count after exclusions: {len(feature_cols)}")
    log(f"Target counts: {train[TARGET].value_counts().to_dict()}")

    for col in feature_cols:
        if train[col].dtype == "object":
            combined = pd.concat([train[col].astype(str), test[col].astype(str)], ignore_index=True)
            codes, _ = pd.factorize(combined)
            train[col] = codes[: len(train)].astype(np.int32)
            test[col] = codes[len(train) :].astype(np.int32)

    train[feature_cols] = train[feature_cols].replace([np.inf, -np.inf], np.nan)
    test[feature_cols] = test[feature_cols].replace([np.inf, -np.inf], np.nan)

    y = train[TARGET].astype(int).to_numpy()
    return train, test, feature_cols, y


def load_v15_train_only() -> tuple[pd.DataFrame, list[str], np.ndarray]:
    train = pd.read_parquet(TRAIN_FILE)
    feature_cols = [c for c in train.columns if c not in EXCLUDE_COLS]
    train[feature_cols] = train[feature_cols].replace([np.inf, -np.inf], np.nan)
    y = train[TARGET].astype(int).to_numpy()
    return train, feature_cols, y


def prepare_linear_matrix(train: pd.DataFrame, feature_cols: list[str]) -> np.ndarray:
    log("Preparing dense float32 matrix with global target-independent median fill + scaling...")
    x_values = train[feature_cols].to_numpy(dtype=np.float32, copy=True)
    del train
    gc.collect()

    for j in range(x_values.shape[1]):
        col = x_values[:, j]
        bad = ~np.isfinite(col)
        if bad.any():
            finite = col[~bad]
            fill = np.median(finite) if finite.size else 0.0
            col[bad] = fill
        mean = float(col.mean())
        std = float(col.std())
        if std > 0:
            col -= mean
            col /= std
        else:
            col -= mean
    return x_values


def save_oof(train: pd.DataFrame, y: np.ndarray, pred: np.ndarray, pred_col: str, filename: str) -> None:
    out = pd.DataFrame({ID_COL: train[ID_COL].to_numpy(), TARGET: y, pred_col: pred})
    out.to_parquet(OOF_DIR / filename, index=False)
    log(f"Saved {OOF_DIR / filename}")


def save_oof_arrays(ids: np.ndarray, y: np.ndarray, pred: np.ndarray, pred_col: str, filename: str) -> None:
    out = pd.DataFrame({ID_COL: ids, TARGET: y, pred_col: pred})
    out.to_parquet(OOF_DIR / filename, index=False)
    log(f"Saved {OOF_DIR / filename}")


def load_linear_matrix_from_disk() -> tuple[np.ndarray, np.ndarray, list[str], np.ndarray]:
    train = pd.read_parquet(TRAIN_FILE)
    feature_cols = [c for c in train.columns if c not in EXCLUDE_COLS]
    ids = train[ID_COL].to_numpy()
    y = train[TARGET].astype(int).to_numpy()
    x_values = prepare_linear_matrix(train, feature_cols)
    return ids, y, feature_cols, x_values


def load_tree_matrix_from_disk() -> tuple[np.ndarray, np.ndarray, list[str], np.ndarray]:
    train = pd.read_parquet(TRAIN_FILE)
    feature_cols = [c for c in train.columns if c not in EXCLUDE_COLS]
    ids = train[ID_COL].to_numpy()
    y = train[TARGET].astype(int).to_numpy()
    log("Preparing tree-model float32 matrix...")
    x_values = train[feature_cols].to_numpy(dtype=np.float32, copy=True)
    del train
    gc.collect()
    x_values[~np.isfinite(x_values)] = np.nan
    return ids, y, feature_cols, x_values


### Logistic Regression and MLP

Both models use the V15 matrix after target-independent median filling/scaling for dense linear/neural-network training.


In [ ]:
def run_lr_from_disk() -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    start = time.time()
    log("\n=== Training Logistic Regression V15 (mini-batch SGD log-loss) ===")
    ids, y, _feature_cols, x_values = load_linear_matrix_from_disk()
    oof = np.zeros(len(y), dtype=np.float32)
    scores = []
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    for fold, (tr_idx, val_idx) in enumerate(skf.split(x_values, y), 1):
        fold_start = time.time()
        rng = np.random.default_rng(SEED + fold)
        model = SGDClassifier(
            loss="log_loss",
            penalty="l2",
            alpha=1e-4,
            learning_rate="optimal",
            average=True,
            random_state=SEED,
        )
        classes = np.array([0, 1], dtype=int)
        batch_size = 8192
        for _epoch in range(12):
            shuffled = tr_idx.copy()
            rng.shuffle(shuffled)
            for start_idx in range(0, len(shuffled), batch_size):
                batch = shuffled[start_idx : start_idx + batch_size]
                model.partial_fit(x_values[batch], y[batch], classes=classes)
        pred = np.zeros(len(val_idx), dtype=np.float32)
        for start_idx in range(0, len(val_idx), batch_size):
            batch = val_idx[start_idx : start_idx + batch_size]
            pred[start_idx : start_idx + len(batch)] = model.predict_proba(x_values[batch])[:, 1]
        oof[val_idx] = pred
        auc = roc_auc_score(y[val_idx], pred)
        scores.append(auc)
        log(f"LR fold {fold}/{N_FOLDS}: AUC={auc:.5f}, time={elapsed(fold_start)}")
        del model
        gc.collect()
    log(f"LR OOF AUC={roc_auc_score(y, oof):.5f}, mean={np.mean(scores):.5f}, std={np.std(scores):.5f}, total={elapsed(start)}")
    del x_values
    gc.collect()
    return ids, y, oof


def run_mlp_from_disk() -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    start = time.time()
    log("\n=== Training MLP V15 ===")
    ids, y, _feature_cols, x_values = load_linear_matrix_from_disk()
    oof = np.zeros(len(y), dtype=np.float32)
    scores = []
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    params = {
        "hidden_layer_sizes": (128, 64),
        "activation": "relu",
        "solver": "adam",
        "alpha": 1e-4,
        "batch_size": 4096,
        "learning_rate_init": 1e-3,
        "max_iter": 1,
        "warm_start": True,
        "early_stopping": False,
        "random_state": SEED,
        "verbose": False,
    }
    for fold, (tr_idx, val_idx) in enumerate(skf.split(x_values, y), 1):
        fold_start = time.time()
        rng = np.random.default_rng(SEED + fold)
        model = MLPClassifier(**params)
        classes = np.array([0, 1], dtype=int)
        batch_size = 4096
        for _epoch in range(20):
            shuffled = tr_idx.copy()
            rng.shuffle(shuffled)
            for start_idx in range(0, len(shuffled), batch_size):
                batch = shuffled[start_idx : start_idx + batch_size]
                model.partial_fit(x_values[batch], y[batch], classes=classes)
        pred = np.zeros(len(val_idx), dtype=np.float32)
        for start_idx in range(0, len(val_idx), batch_size):
            batch = val_idx[start_idx : start_idx + batch_size]
            pred[start_idx : start_idx + len(batch)] = model.predict_proba(x_values[batch])[:, 1]
        oof[val_idx] = pred
        auc = roc_auc_score(y[val_idx], pred)
        scores.append(auc)
        log(f"MLP fold {fold}/{N_FOLDS}: AUC={auc:.5f}, time={elapsed(fold_start)}")
        del model
        gc.collect()
    log(f"MLP OOF AUC={roc_auc_score(y, oof):.5f}, mean={np.mean(scores):.5f}, std={np.std(scores):.5f}, total={elapsed(start)}")
    del x_values
    gc.collect()
    return ids, y, oof


### LightGBM single seed and XGBoost

The single-seed LightGBM and XGBoost baselines use the same V15 folds and save OOF prediction vectors for comparison.


In [ ]:
import lightgbm as lgb
import xgboost as xgb

def run_lgb_single_from_disk() -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    start = time.time()
    log("\n=== Training LightGBM V15 single seed ===")
    ids, y, feature_cols, x_values = load_tree_matrix_from_disk()
    cat_features = [feature_cols.index(c) for c in feature_cols if c in FORCE_CATEGORICAL]
    oof = np.zeros(len(y), dtype=np.float32)
    scores = []
    params = {
        "objective": "binary",
        "metric": "auc",
        "boosting_type": "gbdt",
        "learning_rate": 0.02,
        "num_leaves": 24,
        "max_depth": 6,
        "min_child_samples": 100,
        "min_child_weight": 10,
        "reg_alpha": 0.1,
        "reg_lambda": 0.1,
        "colsample_bytree": 0.7,
        "subsample": 0.8,
        "subsample_freq": 1,
        "max_bin": 63,
        "force_col_wise": True,
        "histogram_pool_size": 256,
        "verbosity": -1,
        "random_state": SEED,
        "n_jobs": 4,
    }
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    for fold, (tr_idx, val_idx) in enumerate(skf.split(x_values, y), 1):
        fold_start = time.time()
        dtrain = lgb.Dataset(x_values[tr_idx], label=y[tr_idx], categorical_feature=cat_features, free_raw_data=True)
        dval = lgb.Dataset(x_values[val_idx], label=y[val_idx], categorical_feature=cat_features, free_raw_data=True)
        model = lgb.train(
            params,
            dtrain,
            num_boost_round=5000,
            valid_sets=[dval],
            valid_names=["valid"],
            callbacks=[lgb.early_stopping(200), lgb.log_evaluation(0)],
        )
        pred = model.predict(x_values[val_idx], num_iteration=model.best_iteration)
        oof[val_idx] = pred
        auc = roc_auc_score(y[val_idx], pred)
        scores.append(auc)
        log(f"LGB fold {fold}/{N_FOLDS}: AUC={auc:.5f}, best_iter={model.best_iteration}, time={elapsed(fold_start)}")
        del model, dtrain, dval
        gc.collect()
    log(f"LGB single OOF AUC={roc_auc_score(y, oof):.5f}, mean={np.mean(scores):.5f}, std={np.std(scores):.5f}, total={elapsed(start)}")
    del x_values
    gc.collect()
    return ids, y, oof


def run_xgb(train: pd.DataFrame, feature_cols: list[str], y: np.ndarray) -> np.ndarray:
    start = time.time()
    log("\n=== Training XGBoost V15 ===")
    x_values = train[feature_cols].to_numpy(dtype=np.float32)
    oof = np.zeros(len(train), dtype=np.float32)
    scores = []
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    for fold, (tr_idx, val_idx) in enumerate(skf.split(x_values, y), 1):
        fold_start = time.time()
        model = xgb.XGBClassifier(
            n_estimators=5000,
            learning_rate=0.02,
            max_depth=6,
            min_child_weight=10,
            subsample=0.8,
            colsample_bytree=0.7,
            max_bin=64,
            reg_alpha=0.1,
            reg_lambda=0.1,
            objective="binary:logistic",
            eval_metric="auc",
            tree_method="hist",
            random_state=SEED,
            n_jobs=4,
            early_stopping_rounds=200,
        )
        model.fit(x_values[tr_idx], y[tr_idx], eval_set=[(x_values[val_idx], y[val_idx])], verbose=False)
        pred = model.predict_proba(x_values[val_idx])[:, 1]
        oof[val_idx] = pred
        auc = roc_auc_score(y[val_idx], pred)
        scores.append(auc)
        log(f"XGB fold {fold}/{N_FOLDS}: AUC={auc:.5f}, best_iter={model.best_iteration}, time={elapsed(fold_start)}")
        del model
        gc.collect()
    log(f"XGB OOF AUC={roc_auc_score(y, oof):.5f}, mean={np.mean(scores):.5f}, std={np.std(scores):.5f}, total={elapsed(start)}")
    return oof


def run_xgb_from_disk() -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    start = time.time()
    log("\n=== Training XGBoost V15 ===")
    ids, y, _feature_cols, x_values = load_tree_matrix_from_disk()
    oof = np.zeros(len(y), dtype=np.float32)
    scores = []
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    for fold, (tr_idx, val_idx) in enumerate(skf.split(x_values, y), 1):
        fold_start = time.time()
        model = xgb.XGBClassifier(
            n_estimators=5000,
            learning_rate=0.02,
            max_depth=6,
            min_child_weight=10,
            subsample=0.8,
            colsample_bytree=0.7,
            max_bin=64,
            reg_alpha=0.1,
            reg_lambda=0.1,
            objective="binary:logistic",
            eval_metric="auc",
            tree_method="hist",
            random_state=SEED,
            n_jobs=4,
            early_stopping_rounds=200,
        )
        model.fit(x_values[tr_idx], y[tr_idx], eval_set=[(x_values[val_idx], y[val_idx])], verbose=False)
        pred = model.predict_proba(x_values[val_idx])[:, 1]
        oof[val_idx] = pred
        auc = roc_auc_score(y[val_idx], pred)
        scores.append(auc)
        log(f"XGB fold {fold}/{N_FOLDS}: AUC={auc:.5f}, best_iter={model.best_iteration}, time={elapsed(fold_start)}")
        del model
        gc.collect()
    log(f"XGB OOF AUC={roc_auc_score(y, oof):.5f}, mean={np.mean(scores):.5f}, std={np.std(scores):.5f}, total={elapsed(start)}")
    del x_values
    gc.collect()
    return ids, y, oof


### CatBoost

CatBoost is included as a V15 baseline model using the same train matrix and target labels.


In [ ]:
from catboost import CatBoostClassifier

def run_catboost(train: pd.DataFrame, feature_cols: list[str], y: np.ndarray) -> np.ndarray:
    start = time.time()
    log("\n=== Training CatBoost V15 ===")
    x_df = train[feature_cols]
    cat_features = [feature_cols.index(c) for c in FORCE_CATEGORICAL if c in feature_cols]
    cat_names = [feature_cols[i] for i in cat_features]
    x_df = x_df.copy()
    for col in cat_names:
        x_df[col] = x_df[col].fillna(-999999).astype(int).astype(str)
    oof = np.zeros(len(train), dtype=np.float32)
    scores = []
    params = {
        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "iterations": 3000,
        "learning_rate": 0.03,
        "depth": 6,
        "l2_leaf_reg": 3.0,
        "min_data_in_leaf": 100,
        "random_strength": 1.0,
        "bagging_temperature": 0.2,
        "border_count": 128,
        "grow_policy": "SymmetricTree",
        "od_type": "Iter",
        "od_wait": 150,
        "random_state": SEED,
        "verbose": False,
        "allow_writing_files": False,
        "task_type": "CPU",
        "thread_count": -1,
    }
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    for fold, (tr_idx, val_idx) in enumerate(skf.split(x_df, y), 1):
        fold_start = time.time()
        model = CatBoostClassifier(**params)
        model.fit(
            x_df.iloc[tr_idx],
            y[tr_idx],
            eval_set=(x_df.iloc[val_idx], y[val_idx]),
            cat_features=cat_features,
            use_best_model=True,
        )
        pred = model.predict_proba(x_df.iloc[val_idx])[:, 1]
        oof[val_idx] = pred
        auc = roc_auc_score(y[val_idx], pred)
        scores.append(auc)
        log(f"CB fold {fold}/{N_FOLDS}: AUC={auc:.5f}, best_iter={model.best_iteration_}, time={elapsed(fold_start)}")
        del model
        gc.collect()
    log(f"CB OOF AUC={roc_auc_score(y, oof):.5f}, mean={np.mean(scores):.5f}, std={np.std(scores):.5f}, total={elapsed(start)}")
    return oof


### LightGBM multi-seed

This is the V15 Multi-Seed LightGBM comparison model (`lgb_v15_ms`) and also one of the two final SE-HC components. During demo, inspect the saved OOF file instead of rerunning this heavy training loop.


In [ ]:

def run_lgb_multiseed_v15_from_disk(seeds=(42, 7, 99)) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Optional heavy cell based on solution_ver02 cell 208.

    Trains LightGBM on V15 for multiple random seeds and returns the averaged OOF vector.
    During the demo, prefer the saved local artifact `outputs/oof_predictions/oof_lgb_v15_multiseed.parquet`.
    """
    import lightgbm as lgb

    start = time.time()
    train, _test, feature_cols, y = load_v15()
    x_df = train[feature_cols]
    categorical_features = [c for c in feature_cols if c in FORCE_CATEGORICAL]

    params = {
        'objective': 'binary',
        'metric': 'auc',
        'boosting_type': 'gbdt',
        'learning_rate': 0.02,
        'num_leaves': 24,
        'max_depth': 6,
        'min_child_samples': 100,
        'min_child_weight': 10,
        'reg_alpha': 0.1,
        'reg_lambda': 0.1,
        'colsample_bytree': 0.7,
        'subsample': 0.8,
        'subsample_freq': 1,
        'max_bin': 255,
        'verbosity': -1,
        'n_jobs': -1,
    }

    seed_oofs = []
    for seed in seeds:
        print(f'Training V15 LightGBM seed={seed}')
        seed_params = dict(params, random_state=seed)
        oof_seed = np.zeros(len(train), dtype=np.float32)
        fold_aucs = []
        skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
        for fold, (tr_idx, val_idx) in enumerate(skf.split(x_df, y), 1):
            fold_start = time.time()
            dtrain = lgb.Dataset(x_df.iloc[tr_idx], label=y[tr_idx], categorical_feature=categorical_features)
            dval = lgb.Dataset(x_df.iloc[val_idx], label=y[val_idx], categorical_feature=categorical_features)
            model = lgb.train(
                seed_params,
                dtrain,
                num_boost_round=5000,
                valid_sets=[dval],
                valid_names=['valid'],
                callbacks=[lgb.early_stopping(200), lgb.log_evaluation(0)],
            )
            pred = model.predict(x_df.iloc[val_idx], num_iteration=model.best_iteration)
            oof_seed[val_idx] = pred
            auc = roc_auc_score(y[val_idx], pred)
            fold_aucs.append(auc)
            print(f'  seed={seed} fold={fold}/{N_FOLDS} AUC={auc:.5f} best_iter={model.best_iteration} time={elapsed(fold_start)}')
            del model, dtrain, dval
            gc.collect()
        print(f'  seed={seed} OOF AUC={roc_auc_score(y, oof_seed):.5f}, mean={np.mean(fold_aucs):.5f}, std={np.std(fold_aucs):.5f}')
        seed_oofs.append(oof_seed)

    oof_ms = np.mean(seed_oofs, axis=0)
    print(f'LightGBM V15 multi-seed OOF AUC={roc_auc_score(y, oof_ms):.5f}, total={elapsed(start)}')
    return train[ID_COL].to_numpy(), y, oof_ms


### Training execution guard

The full rerun path is disabled by default. Use the saved OOF artifacts for inspection and video demo.


In [ ]:

RUN_FULL_BASELINE_WORKFLOW = False

if RUN_FULL_BASELINE_WORKFLOW:
    # This optional path can retrain missing V15 baseline OOFs and write local artifacts.
    # Keep it disabled for GitHub inspection and video demo runs.
    raise NotImplementedError(
        'Use src/train_v15_baselines.py for a full controlled rerun, or enable individual model cells explicitly.'
    )
else:
    print('Full V15 baseline retraining is disabled. This notebook inspects saved local OOF artifacts by default.')


Full V15 baseline retraining is disabled. This notebook inspects saved local OOF artifacts by default.


## Section 4 - V15 baseline comparison table

This table is the corrected V15 baseline comparison. It contains only the six algorithm-level baseline models trained or evaluated on the final V15 feature representation. It intentionally excludes `SPC`, `stack_v8`, and `SE-HC`, because those are ensemble/candidate artifacts rather than simple baseline models.


In [ ]:

def ks_stat(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    fpr, tpr, _ = roc_curve(y_true, y_pred)
    return float(np.max(tpr - fpr))


def lift_at_k(y_true: np.ndarray, y_pred: np.ndarray, k: float = 0.1) -> float:
    n_top = int(len(y_true) * k)
    top_idx = np.argsort(y_pred)[::-1][:n_top]
    return float(y_true[top_idx].mean() / y_true.mean())


def ece_metric(y_true: np.ndarray, y_pred: np.ndarray, n_bins: int = 15) -> float:
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        mask = (y_pred >= bins[i]) & (y_pred < bins[i + 1])
        if mask.sum() > 0:
            ece += abs(y_pred[mask].mean() - y_true[mask].mean()) * mask.sum() / len(y_true)
    return float(ece)


def prediction_column(df: pd.DataFrame) -> str:
    return [c for c in df.columns if c not in [ID_COL, TARGET]][0]


def metric_row(name: str, y_true: np.ndarray, oof: np.ndarray) -> dict[str, float | str]:
    fpr, tpr, thresholds = roc_curve(y_true, oof)
    t_star = thresholds[np.argmax(tpr - fpr)]
    y_pred_binary = (oof >= t_star).astype(int)
    auc = roc_auc_score(y_true, oof)
    return {
        'Model': name,
        'ROC-AUC': auc,
        'Gini': 2 * auc - 1,
        'KS': ks_stat(y_true, oof),
        'PR-AUC': average_precision_score(y_true, oof),
        'Lift@10%': lift_at_k(y_true, oof, 0.1),
        'F1 (t*)': f1_score(y_true, y_pred_binary),
        'Precision': precision_score(y_true, y_pred_binary),
        'Recall': recall_score(y_true, y_pred_binary),
        'Brier': brier_score_loss(y_true, oof),
        'ECE': ece_metric(y_true, oof),
        't*': t_star,
    }


In [ ]:

def build_v15_baseline_comparison() -> pd.DataFrame:
    rows = []
    for model_name, (filename, expected_pred_col) in BASELINE_OOF_FILES.items():
        path = OOF_DIR / filename
        if not path.exists():
            rows.append({'Model': model_name, 'Status': 'MISSING', 'OOF path': str(path.relative_to(ROOT))})
            continue
        df = pd.read_parquet(path)
        pred_col = expected_pred_col if expected_pred_col in df.columns else prediction_column(df)
        y = df[TARGET].astype(int).to_numpy()
        pred = df[pred_col].astype(float).to_numpy()
        row = metric_row(model_name, y, pred)
        row.update({
            'Status': 'OK',
            'OOF path': str(path.relative_to(ROOT)),
            'Shape': f'{df.shape[0]} x {df.shape[1]}',
            'Prediction column': pred_col,
        })
        rows.append(row)
    return pd.DataFrame(rows)

baseline_comparison = build_v15_baseline_comparison()
baseline_comparison[['Model', 'ROC-AUC', 'Gini', 'KS', 'PR-AUC', 'Lift@10%', 'F1 (t*)', 'Precision', 'Recall', 'Brier', 'ECE', 't*', 'Status']]


               Model  ROC-AUC     Gini       KS   PR-AUC  Lift@10%  F1 (t*)  Precision   Recall    Brier      ECE       t* Status
 Logistic Regression 0.782546 0.565092 0.424540 0.267877  3.658492 0.286685   0.179690 0.708640 0.070328 0.061353 0.155053     OK
                 MLP 0.725532 0.451063 0.337656 0.204363  3.043372 0.247828   0.152609 0.659013 0.075444 0.043269 0.045007     OK
LightGBM single seed 0.798504 0.597009 0.452538 0.295101  3.898981 0.298229   0.187321 0.731078 0.065093 0.002370 0.076887     OK
 LightGBM multi-seed 0.799749 0.599497 0.453514 0.297331  3.898175 0.292224   0.181390 0.751259 0.064972 0.002716 0.072577     OK
             XGBoost 0.798955 0.597910 0.452732 0.296833  3.912274 0.302721   0.191730 0.718872 0.065011 0.002048 0.081166     OK
            CatBoost 0.798170 0.596340 0.452687 0.295139  3.907440 0.293298   0.182520 0.746183 0.065092 0.003858 0.073554     OK

## Section 5 - Ensemble candidate prediction library

This section summarizes OOF prediction candidates that are later consumed by the SE-HC construction notebook. These candidates are not all baseline models; some are stacked or historical predictors used to increase ensemble diversity.

`SE_HC_SELECTED_CANDIDATES` and `SE_HC_SELECTED_CANDIDATES_MS` are additional experiments only and are not presented here as final thesis results.


In [ ]:

def summarize_candidate_library() -> pd.DataFrame:
    rows = []
    for name, (filename, expected_pred_col) in CANDIDATE_OOF_FILES.items():
        path = OOF_DIR / filename
        if not path.exists():
            rows.append({'Candidate predictor': name, 'Status': 'MISSING', 'OOF path': str(path.relative_to(ROOT))})
            continue
        df = pd.read_parquet(path)
        pred_col = expected_pred_col if expected_pred_col in df.columns else prediction_column(df)
        rows.append({
            'Candidate predictor': name,
            'OOF path': str(path.relative_to(ROOT)),
            'Shape': f'{df.shape[0]} x {df.shape[1]}',
            'Prediction column': pred_col,
            'OOF ROC-AUC': roc_auc_score(df[TARGET].astype(int), df[pred_col].astype(float)),
            'Status': 'OK',
        })
    return pd.DataFrame(rows)

candidate_library = summarize_candidate_library()
candidate_library


         Candidate predictor                                              OOF path      Shape Prediction column  OOF ROC-AUC Status
              SPC / fp_final          outputs/oof_predictions/oof_FP_FINAL.parquet 307507 x 3         oof_final     0.799879     OK
                  lgb_v15_ms outputs/oof_predictions/oof_lgb_v15_multiseed.parquet 307507 x 3           oof_lgb     0.799749     OK
                  lgb_v12_ms outputs/oof_predictions/oof_lgb_v12_multiseed.parquet 307507 x 3           oof_lgb     0.798544     OK
                  lgb_v11_ms outputs/oof_predictions/oof_lgb_v11_multiseed.parquet 307507 x 3           oof_lgb     0.797993     OK
                   lgb_v7_ms  outputs/oof_predictions/oof_lgb_v7_multiseed.parquet 307507 x 3           oof_lgb     0.797122     OK
                   cb_v11_ms  outputs/oof_predictions/oof_cb_v11_multiseed.parquet 307507 x 3            oof_cb     0.796316     OK
                     xgb_v11           outputs/oof_predictions/oof_xgb_v11.p

## Section 6 - Outputs handed off to ensemble stage

The ensemble construction notebook consumes saved OOF vectors and metrics from this stage. Full OOF parquet files are local-only artifacts and are not committed to GitHub.


In [ ]:

handoff = {
    'baseline_oofs': {name: str((OOF_DIR / filename).relative_to(ROOT)) for name, (filename, _) in BASELINE_OOF_FILES.items()},
    'candidate_oofs': {name: str((OOF_DIR / filename).relative_to(ROOT)) for name, (filename, _) in CANDIDATE_OOF_FILES.items()},
    'corrected_baseline_table': 'computed in this notebook from canonical V15 OOF parquet files',
    'existing_metric_csv': str((METRICS_DIR / 'final_v15_baseline_comparison.csv').relative_to(ROOT)),
    'existing_metric_csv_note': 'local summary may include ensemble/candidate rows; Section 4 is the corrected baseline-only view',
    'final_se_hc_metrics': str((REPORTS_DIR / 'metrics_SE_HC.json').relative_to(ROOT)),
    'note': 'Full OOF parquet files are local-only artifacts and are not committed to GitHub.',
}
handoff

{'baseline_oofs': {'Logistic Regression': 'outputs/oof_predictions/oof_lr_v15_baseline.parquet', 'MLP': 'outputs/oof_predictions/oof_mlp_v15.parquet', 'LightGBM single seed': 'outputs/oof_predictions/oof_lgb_v15_single_seed.parquet', 'LightGBM multi-seed': 'outputs/oof_predictions/oof_lgb_v15_multiseed.parquet', 'XGBoost': 'outputs/oof_predictions/oof_xgb_v15.parquet', 'CatBoost': 'outputs/oof_predictions/oof_cb_v15.parquet'}, 'candidate_oofs': {'SPC / fp_final': 'outputs/oof_predictions/oof_FP_FINAL.parquet', 'lgb_v15_ms': 'outputs/oof_predictions/oof_lgb_v15_multiseed.parquet', 'lgb_v12_ms': 'outputs/oof_predictions/oof_lgb_v12_multiseed.parquet', 'lgb_v11_ms': 'outputs/oof_predictions/oof_lgb_v11_multiseed.parquet', 'lgb_v7_ms': 'outputs/oof_predictions/oof_lgb_v7_multiseed.parquet', 'cb_v11_ms': 'outputs/oof_predictions/oof_cb_v11_multiseed.parquet', 'xgb_v11': 'outputs/oof_predictions/oof_xgb_v11.parquet', 'stack_v8 / K3_stack_v8_FINAL': 'outputs/oof_predictions/oof_K3_stack_v8_FI